In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_deliveries = pd.read_csv('CSV/deliveries.csv')
df_matches = pd.read_csv('CSV/matches.csv')

In [3]:
df_matches['season'].value_counts()

season
2013       76
2012       74
2022       74
2023       74
2011       73
2024       71
2009/10    60
2014       60
2016       60
2018       60
2019       60
2020/21    60
2021       60
2015       59
2017       59
2007/08    58
2009       57
Name: count, dtype: int64

In [4]:
df_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   str    
 2   city             1044 non-null   str    
 3   date             1095 non-null   str    
 4   match_type       1095 non-null   str    
 5   player_of_match  1090 non-null   str    
 6   venue            1095 non-null   str    
 7   team1            1095 non-null   str    
 8   team2            1095 non-null   str    
 9   toss_winner      1095 non-null   str    
 10  toss_decision    1095 non-null   str    
 11  winner           1090 non-null   str    
 12  result           1095 non-null   str    
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   str    
 17  method           21 non-n

In [5]:
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bengaluru',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

In [6]:
name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['team1', 'team2', 'winner']:
    df_matches[col] = df_matches[col].replace(name_map)


for col in ['batting_team', 'bowling_team']:
    df_deliveries[col] = df_deliveries[col].replace(name_map)


In [7]:
df_matches = df_matches[df_matches['team1'].isin(teams)]
df_matches = df_matches[df_matches['team2'].isin(teams)]

df_deliveries = df_deliveries[df_deliveries['batting_team'].isin(teams)]
df_deliveries = df_deliveries[df_deliveries['bowling_team'].isin(teams)]

In [8]:
df_matches = df_matches[df_matches['method'].isna()]

In [9]:
df_matches['venue_clean'] = (
    df_matches['venue']
    .str.strip()
    .str.lower()
    .str.split(",").str[0]
)

In [10]:
venue_mapping = {
    # Delhi
    "feroz shah kotla": "arun jaitley stadium",
    "arun jaitley stadium": "arun jaitley stadium",

    # Ahmedabad
    "sardar patel stadium": "narendra modi stadium",
    "motera": "narendra modi stadium",
    "narendra modi stadium": "narendra modi stadium",

    # Mohali
    "punjab cricket association stadium": "punjab cricket association is bindra stadium",
    "punjab cricket association is bindra stadium": "punjab cricket association is bindra stadium",

    # Pune
    "subrata roy sahara stadium": "maharashtra cricket association stadium",
    "maharashtra cricket association stadium": "maharashtra cricket association stadium",

    # Abu Dhabi
    "sheikh zayed stadium": "zayed cricket stadium",
    "zayed cricket stadium": "zayed cricket stadium",

    # Chennai
    "ma chidambaram stadium": "ma chidambaram stadium",

    # Mumbai
    "wankhede stadium": "wankhede stadium",
    "brabourne stadium": "brabourne stadium",
    "dr dy patil sports academy": "dr dy patil sports academy",

    # Hyderabad
    "rajiv gandhi international stadium": "rajiv gandhi international stadium",

    # Visakhapatnam
    "dr. y.s. rajasekhara reddy aca-vdca cricket stadium": "dr. y.s. rajasekhara reddy aca-vdca cricket stadium",

    # Jaipur
    "sawai mansingh stadium": "sawai mansingh stadium",

    # Nagpur
    "vidarbha cricket association stadium": "vidarbha cricket association stadium",

    # Lucknow
    "bharat ratna shri atal bihari vajpayee ekana cricket stadium": "bharat ratna shri atal bihari vajpayee ekana cricket stadium",

    # Ranchi
    "jsca international stadium complex": "jsca international stadium complex",

    # Raipur
    "shaheed veer narayan singh international stadium": "shaheed veer narayan singh international stadium",

    # Indore
    "holkar cricket stadium": "holkar cricket stadium",

    # Bangalore
    "m chinnaswamy stadium": "m chinnaswamy stadium",

    # Kolkata
    "eden gardens": "eden gardens",

    # International (South Africa / UAE)
    "newlands": "newlands",
    "kingsmead": "kingsmead",
    "supersport park": "supersport park",
    "st george's park": "st george's park",
    "new wanderers stadium": "new wanderers stadium",
    "buffalo park": "buffalo park",
    "outsurance oval": "outsurance oval",
    "de beers diamond oval": "de beers diamond oval",
    "barsapara cricket stadium": "barsapara cricket stadium",
    "sharjah cricket stadium": "sharjah cricket stadium",
    "dubai international cricket stadium": "dubai international cricket stadium",
    "maharaja yadavindra singh international cricket stadium": "maharaja yadavindra singh international cricket stadium",
}


In [11]:
df_matches['venue_clean'] = (
    df_matches['venue_clean']
    .str.strip().str.lower()
    .replace(venue_mapping)
)


In [12]:
df_matches.drop('venue', axis=1,inplace=True)
df_matches.rename(columns={'venue_clean':'venue'},inplace=True)

In [13]:
df_matches = df_matches[['id','venue','winner','target_runs']]

In [14]:
df_matches.rename(columns={'id' : 'match_id'},inplace=True)

In [15]:
print(df_matches['match_id'].nunique())
print(df_deliveries['match_id'].nunique())

963
980


In [16]:
delivery = df_matches.merge(df_deliveries, on='match_id')

In [17]:
delivery = delivery[delivery['inning'] == 2]

In [18]:
delivery['current_score'] = delivery.groupby('match_id')['total_runs'].cumsum()

In [19]:
delivery['runs_left'] =  delivery['target_runs'] - delivery['current_score']

In [20]:
delivery['balls_left'] =  120 - (delivery['over'] * 6 + delivery['ball'])

In [21]:
wickets = delivery.groupby('match_id')['is_wicket'].cumsum()
delivery['wickets_left'] = 10 - wickets

In [22]:
delivery['crr'] = delivery['current_score'] * 6 / (120- delivery['balls_left'])

In [23]:
delivery['rrr'] = delivery['runs_left'] * 6 / delivery['balls_left']

In [24]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [25]:
valid_balls = delivery[delivery['extras_type'].isna() | 
                       delivery['extras_type'].isin(['legbyes', 'byes', 'noballs'])]


In [26]:
per_match = (
    delivery.groupby(['batter', 'match_id']).agg(
        runs_in_match=('batsman_runs', 'sum'),
        balls_in_match=('ball', 'count')
    )
    .reset_index()
)


In [27]:
per_match = per_match.sort_values(['batter','match_id'])

In [28]:
per_match['overall_runs'] = (
    per_match.groupby('batter')['runs_in_match']
    .transform(lambda x: x.cumsum().shift(fill_value=0))
)
per_match['overall_balls'] = (
    per_match.groupby('batter')['balls_in_match']
    .transform(lambda x: x.cumsum().shift(fill_value=0))
)


In [29]:
per_match['overall_SR'] = ((per_match['overall_runs'] / per_match['overall_balls']) * 100).round(2)

In [30]:
per_match["last5_runs"] = (
    per_match.groupby("batter")["runs_in_match"]
    .transform(lambda x: x.shift(fill_value = 0).rolling(5, min_periods=1).sum())
)

per_match["last5_balls"] = (
    per_match.groupby("batter")["balls_in_match"]
    .transform(lambda x: x.shift(fill_value = 0).rolling(5, min_periods=1).sum())
)


In [31]:
per_match['last5_SR'] = ((per_match['last5_runs'] / per_match['last5_balls']) * 100).round(2)

In [32]:
per_match[['overall_SR','last5_SR']] = per_match[['overall_SR','last5_SR']].fillna(0)

In [33]:
per_match

,batter,match_id,runs_in_match,balls_in_match,overall_runs,overall_balls,overall_SR,last5_runs,last5_balls,last5_SR
0,A Ashish Reddy,548352,3,3,0,0,0.00,0.0,0.0,0.00
1,A Ashish Reddy,548359,8,8,3,3,100.00,3.0,3.0,100.00
2,A Ashish Reddy,548373,10,4,11,11,100.00,11.0,11.0,100.00
3,A Ashish Reddy,598004,14,12,21,15,140.00,21.0,15.0,140.00
4,A Ashish Reddy,598010,16,9,35,27,129.63,35.0,27.0,129.63
...,...,...,...,...,...,...,...,...,...,...
7197,Z Khan,729317,1,2,49,66,74.24,32.0,38.0,84.21
7198,Z Khan,980993,2,4,50,68,73.53,33.0,37.0,89.19
7199,Z Khan,1082595,1,1,52,72,72.22,14.0,20.0,70.00
7200,Z Khan,1082635,2,11,53,73,72.60,4.0,14.0,28.57


In [34]:
delivery

,match_id,venue,winner,target_runs,inning,batting_team,bowling_team,over,ball,batter,...,is_wicket,player_dismissed,dismissal_kind,fielder,current_score,runs_left,balls_left,wickets_left,crr,rrr
124,335982,m chinnaswamy stadium,Kolkata Knight Riders,223.0,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,1,R Dravid,...,0,NaN,NaN,NaN,1,222.0,119,10,6.000000,11.193277
125,335982,m chinnaswamy stadium,Kolkata Knight Riders,223.0,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,2,W Jaffer,...,0,NaN,NaN,NaN,2,221.0,118,10,6.000000,11.237288
126,335982,m chinnaswamy stadium,Kolkata Knight Riders,223.0,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,3,W Jaffer,...,0,NaN,NaN,NaN,2,221.0,117,10,4.000000,11.333333
127,335982,m chinnaswamy stadium,Kolkata Knight Riders,223.0,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,4,W Jaffer,...,0,NaN,NaN,NaN,3,220.0,116,10,4.500000,11.379310
128,335982,m chinnaswamy stadium,Kolkata Knight Riders,223.0,2,Royal Challengers Bengaluru,Kolkata Knight Riders,0,5,R Dravid,...,0,NaN,NaN,NaN,4,219.0,115,10,4.800000,11.426087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230977,1426312,ma chidambaram stadium,Kolkata Knight Riders,114.0,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,5,SS Iyer,...,0,NaN,NaN,NaN,110,4.0,61,8,11.186441,0.393443
230978,1426312,ma chidambaram stadium,Kolkata Knight Riders,114.0,2,Kolkata Knight Riders,Sunrisers Hyderabad,9,6,VR Iyer,...,0,NaN,NaN,NaN,111,3.0,60,8,11.100000,0.300000
230979,1426312,ma chidambaram stadium,Kolkata Knight Riders,114.0,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,1,VR Iyer,...,0,NaN,NaN,NaN,112,2.0,59,8,11.016393,0.203390
230980,1426312,ma chidambaram stadium,Kolkata Knight Riders,114.0,2,Kolkata Knight Riders,Sunrisers Hyderabad,10,2,SS Iyer,...,0,NaN,NaN,NaN,113,1.0,58,8,10.935484,0.103448


In [35]:
delivery = delivery = delivery.merge(
    per_match,
    on = ['match_id','batter'],
    how = 'left'
)

In [36]:
delivery['result'] = delivery.apply(result,axis=1)

In [37]:
delivery.columns

Index(['match_id', 'venue', 'winner', 'target_runs', 'inning', 'batting_team',
       'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker',
       'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket',
       'player_dismissed', 'dismissal_kind', 'fielder', 'current_score',
       'runs_left', 'balls_left', 'wickets_left', 'crr', 'rrr',
       'runs_in_match', 'balls_in_match', 'overall_runs', 'overall_balls',
       'overall_SR', 'last5_runs', 'last5_balls', 'last5_SR', 'result'],
      dtype='str')

In [38]:
final_df = delivery[['match_id','batting_team','bowling_team','venue','runs_left','balls_left','wickets_left','target_runs','crr','rrr','overall_runs', 'overall_balls',
       'overall_SR', 'last5_runs', 'last5_balls', 'last5_SR','result']]

In [39]:
final_df = final_df[final_df['balls_left'] != 0]

In [40]:
final_df.dropna(subset=['rrr'],inplace=True)

In [41]:
final_df.columns

Index(['match_id', 'batting_team', 'bowling_team', 'venue', 'runs_left',
       'balls_left', 'wickets_left', 'target_runs', 'crr', 'rrr',
       'overall_runs', 'overall_balls', 'overall_SR', 'last5_runs',
       'last5_balls', 'last5_SR', 'result'],
      dtype='str')

In [42]:
print(final_df.head(1))

   match_id                 batting_team           bowling_team  \
0    335982  Royal Challengers Bengaluru  Kolkata Knight Riders   

                   venue  runs_left  balls_left  wickets_left  target_runs  \
0  m chinnaswamy stadium      222.0         119            10        223.0   

   crr        rrr  overall_runs  overall_balls  overall_SR  last5_runs  \
0  6.0  11.193277             0              0         0.0         0.0   

   last5_balls  last5_SR  result  
0          0.0       0.0       0  


In [43]:
def phase_and_progress(row, total_overs=20):
    balls_bowled = total_overs*6 - row['balls_left']
    
    if row['balls_left'] <= 30:
        phase = 'death_over'
    elif row['balls_left'] >= 84:
        phase = 'powerplay'
    else:
        phase = 'middle_over'
    
    overs_completed = balls_bowled / 6
    
    return pd.Series([phase, overs_completed], index=['phase', 'overs_completed'])

In [44]:
final_df[['current_phase','over_completed']] = final_df.apply(phase_and_progress, axis=1)

In [45]:
final_df

,match_id,batting_team,bowling_team,venue,runs_left,balls_left,wickets_left,target_runs,crr,rrr,overall_runs,overall_balls,overall_SR,last5_runs,last5_balls,last5_SR,result,current_phase,over_completed
0,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,m chinnaswamy stadium,222.0,119,10,223.0,6.000000,11.193277,0,0,0.00,0.0,0.0,0.00,0,powerplay,0.166667
1,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,m chinnaswamy stadium,221.0,118,10,223.0,6.000000,11.237288,0,0,0.00,0.0,0.0,0.00,0,powerplay,0.333333
2,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,m chinnaswamy stadium,221.0,117,10,223.0,4.000000,11.333333,0,0,0.00,0.0,0.0,0.00,0,powerplay,0.500000
3,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,m chinnaswamy stadium,220.0,116,10,223.0,4.500000,11.379310,0,0,0.00,0.0,0.0,0.00,0,powerplay,0.666667
4,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,m chinnaswamy stadium,219.0,115,10,223.0,4.800000,11.426087,0,0,0.00,0.0,0.0,0.00,0,powerplay,0.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111716,1426312,Kolkata Knight Riders,Sunrisers Hyderabad,ma chidambaram stadium,4.0,61,8,114.0,11.186441,0.393443,1350,1118,120.75,218.0,144.0,151.39,1,middle_over,9.833333
111717,1426312,Kolkata Knight Riders,Sunrisers Hyderabad,ma chidambaram stadium,3.0,60,8,114.0,11.100000,0.300000,641,487,131.62,160.0,102.0,156.86,1,middle_over,10.000000
111718,1426312,Kolkata Knight Riders,Sunrisers Hyderabad,ma chidambaram stadium,2.0,59,8,114.0,11.016393,0.203390,641,487,131.62,160.0,102.0,156.86,1,middle_over,10.166667
111719,1426312,Kolkata Knight Riders,Sunrisers Hyderabad,ma chidambaram stadium,1.0,58,8,114.0,10.935484,0.103448,1350,1118,120.75,218.0,144.0,151.39,1,middle_over,10.333333


In [46]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [47]:
unique_matches = final_df['match_id'].unique()
train_matches, test_matches = train_test_split(
    unique_matches, test_size=0.2, random_state=2
)

In [48]:
train_data = final_df[final_df['match_id'].isin(train_matches)]
test_data = final_df[final_df['match_id'].isin(test_matches)]

In [49]:
train_data['match_id'].value_counts().sort_values()

match_id
1178424     20
1082626     49
598068      50
1136608     51
1254093     53
          ... 
1304076    131
1426268    132
829737     133
829811     133
1359480    135
Name: count, Length: 768, dtype: int64

In [50]:
test_data['match_id'].value_counts().sort_values()

match_id
829813       7
336021      38
1304082     49
829803      63
1254087     65
          ... 
1359537    129
1216522    130
829777     130
1426294    130
1370350    131
Name: count, Length: 193, dtype: int64

In [51]:
train_data = train_data.sample(frac = 1, random_state=42).reset_index(drop=True)
test_data = test_data.sample(frac = 1, random_state=42).reset_index(drop=True)

In [52]:
X_train = train_data.drop(columns=['result','match_id'])
y_train = train_data['result']

X_test = test_data.drop(columns=['result','match_id'])
y_test = test_data['result']

In [53]:
trf = ColumnTransformer(
    transformers=[
        ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'venue','current_phase'])
    ], remainder='passthrough'
)

In [54]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver ='liblinear'))
])

In [55]:
pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('step1', ...), ('step2', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('trf', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse 

In [56]:
scores = cross_val_score(pipe,X_train,y_train,cv = 10, scoring = 'accuracy' )
print(f'CV Accuracy scores: {scores}')
print(f'mean : {scores.mean()}')

CV Accuracy scores: [0.82029764 0.80698221 0.80843684 0.81280072 0.80496811 0.81268882
 0.80933199 0.80619895 0.81324829 0.814906  ]
mean : 0.810985956204523


In [57]:
y_pred = pipe.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("Classification Report:\n", classification_report(y_test, y_pred))


Test Accuracy: 0.7471316700054634
Confusion Matrix:
 [[7979 2954]
 [2600 8431]]
Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.73      0.74     10933
           1       0.74      0.76      0.75     11031

    accuracy                           0.75     21964
   macro avg       0.75      0.75      0.75     21964
weighted avg       0.75      0.75      0.75     21964



In [58]:
X_train.columns

Index(['batting_team', 'bowling_team', 'venue', 'runs_left', 'balls_left',
       'wickets_left', 'target_runs', 'crr', 'rrr', 'overall_runs',
       'overall_balls', 'overall_SR', 'last5_runs', 'last5_balls', 'last5_SR',
       'current_phase', 'over_completed'],
      dtype='str')

In [59]:
def build_match_features(batting_team, bowling_team, balls_bowled,current_runs, target_runs, wickets_left, striker, venue, player_data):

    balls_left = 120 - balls_bowled
    runs_left = target_runs - current_runs
    overs_completed = balls_bowled / 6

    crr = current_runs * 6 / (120- balls_left)
    rrr = runs_left * 6 / balls_left

    striker_stats = player_data[player_data["batter"] == striker].iloc[-1]
    overall_runs = striker_stats["overall_runs"]
    overall_balls = striker_stats["overall_balls"]
    overall_SR = striker_stats["overall_SR"]
    last5_runs = striker_stats["last5_runs"]
    last5_balls = striker_stats["last5_balls"]
    last5_SR = striker_stats["last5_SR"]


    if balls_left <= 30:
        phase = 'death_over'
    elif balls_left >= 84:
        phase = 'powerplay'
    else:
        phase = 'middle_over'


    data = {
        "batting_team": [batting_team],
        "bowling_team": [bowling_team],
        "venue": [venue],
        "runs_left": [runs_left],
        "balls_left": [balls_left],
        "wickets_left": [wickets_left],
        "target_runs": [target_runs],
        "crr": [crr],
        "rrr": [rrr],
        "overall_runs": [overall_runs],
        "overall_balls": [overall_balls],
        "overall_SR": [overall_SR],
        "last5_runs": [last5_runs],
        "last5_balls": [last5_balls],
        "last5_SR": [last5_SR],
        "current_phase": [phase],
        "over_completed": [overs_completed],
    }

    return pd.DataFrame(data)


In [60]:
match_scenario = build_match_features(
    batting_team="Royal Challengers Bengaluru",
    bowling_team="Delhi Capitals",
    balls_bowled=96,
    current_runs=140,
    target_runs=180,
    wickets_left=5,
    striker="F du Plessis",
    venue="m chinnaswamy stadium",
    player_data=per_match
)


In [61]:
probability = pipe.predict_proba(match_scenario)

In [62]:
print(f'Batting team %: {probability[0][1]}  Bowling team%: {probability[0][0]}')

Batting team %: 0.5650050685125024  Bowling team%: 0.43499493148749757


In [63]:
import pickle

In [64]:
delivery['batter'].value_counts()

batter
V Kohli           2526
RG Sharma         2257
S Dhawan          2150
DA Warner         2090
G Gambhir         1877
                  ... 
Abdul Basith         1
VG Arora             1
SZ Mulani            1
Mayank Dagar         1
Saurav Chauhan       1
Name: count, Length: 594, dtype: int64

In [65]:
with open("ipl_model.pkl", "wb") as f:
    pickle.dump(pipe, f)

print("Model exported as ipl_model.pkl ")

Model exported as ipl_model.pkl 


In [66]:
import joblib

In [67]:
with open('ipl_model.joblib', 'wb') as f:
    joblib.dump(pipe, f)

print("Model exported as ipl_model.joblib ")

Model exported as ipl_model.joblib 
